# 13h CLIP-SENet CityFlow Fusion (FT)

Extract CityFlowV2 tracklet features with the 13f fine-tuned CLIP-SENet checkpoint, align them to the existing 10b TransReID+DINOv2 embedding index, and run the locked Stage4/Stage5 fusion sweep.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

%pip install -q --upgrade --index-url https://download.pytorch.org/whl/cu124 torch==2.4.1+cu124 torchvision==0.19.1+cu124
%pip install -q --upgrade open_clip_torch==2.30.0 timm==1.0.11 pretrainedmodels==0.7.4 tqdm

In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import tarfile
import time
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch

WORK_DIR = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
PROJECT = WORK_DIR / "gp"
REPO_URL = "https://github.com/MRKDaGods/gp.git"
BRANCH = "feature/pretrained-ensemble"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


if not PROJECT.exists():
    print(f"Cloning {REPO_URL} ({BRANCH})")
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(PROJECT)])
else:
    print("Repo already present, pulling latest")
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))

try:
    import faiss
    print(f"faiss ok ({faiss.__version__})")
except ImportError:
    try:
        pip("faiss-cpu")
    except Exception:
        pip("faiss-gpu")

try:
    import trackeval
    print("trackeval ok")
except ImportError:
    pip("git+https://github.com/JonathonLuiten/TrackEval.git")

pip("motmetrics", "loguru", "omegaconf", "rich", "networkx>=3.1", "click", "numpy", "scipy", "pandas", "scikit-learn")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"], cwd=str(PROJECT))

print(f"Python: {sys.version.split()[0]}")
print(f"Torch: {torch.__version__}, CUDA={torch.cuda.is_available()}, arch={torch.cuda.get_arch_list() if torch.cuda.is_available() else None}")
print(f"Project: {PROJECT}")
print(f"Device: {DEVICE}")

In [ ]:
"""CLIP-SENet architecture for vehicle re-identification.

Milestone M1 covers only the model port and local forward-pass smoke tests.
Training losses, camera/viewpoint embeddings, and Kaggle integration are
intentionally deferred.
"""

from __future__ import annotations

from dataclasses import dataclass

import logging
import torch
import torch.nn as nn
import torch.nn.functional as F

logger = logging.getLogger(__name__)
if not logger.handlers:
    _h = logging.StreamHandler()
    _h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(_h)
    logger.setLevel(logging.INFO)


@dataclass(frozen=True)
class LoadedBackboneInfo:
    """Describes the backbone variant that was successfully loaded."""

    family: str
    model_name: str
    pretrained_tag: str | None = None


class AFEMBlock(nn.Module):
    """Adaptive Fine-grained Enhancement Module.

    This implements the paper's ambiguous Eq. (4) using the `(G + 1)`
    interpretation: `G` grouped weighted residual chunks plus one identity
    residual path. Set `residual_mode="sum_only"` to drop the identity term and
    return only the weighted grouped sum.
    """

    def __init__(
        self,
        in_dim: int = 2048,
        out_dim: int = 2048,
        num_groups: int = 32,
        residual_mode: str = "grouped_identity",
    ):
        super().__init__()
        if out_dim % num_groups != 0:
            raise ValueError(
                f"AFEM out_dim={out_dim} must be divisible by num_groups={num_groups}"
            )
        if residual_mode not in {"grouped_identity", "sum_only"}:
            raise ValueError(
                "residual_mode must be 'grouped_identity' or 'sum_only'"
            )

        self.in_dim = in_dim
        self.out_dim = out_dim
        self.num_groups = num_groups
        self.group_dim = out_dim // num_groups
        self.residual_mode = residual_mode

        self.shared = nn.Sequential(
            nn.Linear(in_dim, out_dim, bias=False),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(inplace=True),
        )
        self.group_weights = nn.Parameter(torch.randn(num_groups, self.group_dim))

        self._reset_parameters()

    def _reset_parameters(self) -> None:
        linear = self.shared[0]
        nn.init.kaiming_normal_(linear.weight, mode="fan_out")
        bn = self.shared[1]
        nn.init.ones_(bn.weight)
        nn.init.zeros_(bn.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.shared(x)
        grouped = h.view(h.shape[0], self.num_groups, self.group_dim)
        weighted = grouped * self.group_weights.unsqueeze(0)
        enhanced = weighted.reshape(h.shape[0], self.out_dim)
        if self.residual_mode == "sum_only":
            return enhanced
        return h + enhanced


class _ResNetFeatureWrapper(nn.Module):
    """Wrap torchvision-style ResNet backbones to expose pooled 2048-d features."""

    def __init__(self, model: nn.Module):
        super().__init__()
        self.conv1 = model.conv1
        self.bn1 = model.bn1
        self.relu = model.relu
        self.maxpool = model.maxpool
        self.layer1 = model.layer1
        self.layer2 = model.layer2
        self.layer3 = model.layer3
        self.layer4 = model.layer4
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        return torch.flatten(x, 1)


class ResNet101IBNBranch(nn.Module):
    """Appearance branch backed by real ResNet101 IBN-a with deterministic fallbacks."""

    _IBN_MODEL = "resnet101_ibn_a"
    _FALLBACK_MODEL = "resnet101"

    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.output_dim = 2048
        self.backbone: nn.Module
        self.loaded_backbone: LoadedBackboneInfo

        for loader in (
            self._load_pretrainedmodels_ibn,
            self._load_torch_hub_ibn,
            self._load_timm_ibn,
            self._load_timm_plain,
        ):
            loaded = loader(pretrained=pretrained)
            if loaded is None:
                continue
            self.backbone, self.loaded_backbone = loaded
            logger.info(
                "Appearance branch loaded via '%s' model='%s' pretrained_tag='%s'",
                self.loaded_backbone.family,
                self.loaded_backbone.model_name,
                self.loaded_backbone.pretrained_tag,
            )
            return

        raise ImportError(
            "Unable to load appearance backbone via pretrainedmodels, torch.hub, or timm"
        )

    def _load_pretrainedmodels_ibn(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            import pretrainedmodels
        except ImportError:
            logger.warning(
                "Appearance branch loader 'pretrainedmodels' is unavailable; trying torch.hub"
            )
            return None

        constructor = getattr(pretrainedmodels, self._IBN_MODEL, None)
        if constructor is None:
            logger.warning(
                "Appearance branch loader 'pretrainedmodels' has no '%s' entry; trying torch.hub",
                self._IBN_MODEL,
            )
            return None

        pretrained_tag = "imagenet" if pretrained else None
        try:
            raw_model = constructor(pretrained=pretrained_tag)
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'pretrainedmodels' failed for '%s': %s",
                self._IBN_MODEL,
                exc,
            )
            return None

        if hasattr(raw_model, "last_linear"):
            raw_model.last_linear = nn.Identity()
        backbone = _ResNetFeatureWrapper(raw_model)
        return backbone, LoadedBackboneInfo(
            family="pretrainedmodels",
            model_name=self._IBN_MODEL,
            pretrained_tag=pretrained_tag or "random_init",
        )

    def _load_torch_hub_ibn(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            raw_model = torch.hub.load(
                "XingangPan/IBN-Net",
                self._IBN_MODEL,
                pretrained=pretrained,
                trust_repo=True,
            )
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'torch.hub' failed for '{}': {}",
                self._IBN_MODEL,
                exc,
            )
            return None

        if hasattr(raw_model, "fc"):
            raw_model.fc = nn.Identity()
        backbone = _ResNetFeatureWrapper(raw_model)
        return backbone, LoadedBackboneInfo(
            family="torch.hub",
            model_name=self._IBN_MODEL,
            pretrained_tag="official_pretrained" if pretrained else "random_init",
        )

    def _load_timm_ibn(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            import timm
        except ImportError as exc:
            raise ImportError("timm is required for ResNet101IBNBranch fallbacks") from exc

        available = set(timm.list_models())
        if self._IBN_MODEL not in available:
            logger.warning(
                "Appearance branch loader 'timm' has no '%s' entry; trying plain '%s'",
                self._IBN_MODEL,
                self._FALLBACK_MODEL,
            )
            return None

        try:
            backbone = timm.create_model(
                self._IBN_MODEL,
                pretrained=pretrained,
                num_classes=0,
                global_pool="avg",
            )
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'timm' failed for '%s': %s",
                self._IBN_MODEL,
                exc,
            )
            return None

        return backbone, LoadedBackboneInfo(
            family="timm",
            model_name=self._IBN_MODEL,
            pretrained_tag="timm_pretrained" if pretrained else "random_init",
        )

    def _load_timm_plain(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            import timm
        except ImportError as exc:
            raise ImportError("timm is required for ResNet101IBNBranch fallbacks") from exc

        try:
            backbone = timm.create_model(
                self._FALLBACK_MODEL,
                pretrained=pretrained,
                num_classes=0,
                global_pool="avg",
            )
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'timm' failed for plain '%s': %s",
                self._FALLBACK_MODEL,
                exc,
            )
            return None

        logger.warning(
            "Appearance branch fell back to plain '%s' because no IBN-a loader succeeded",
            self._FALLBACK_MODEL,
        )
        return backbone, LoadedBackboneInfo(
            family="timm",
            model_name=self._FALLBACK_MODEL,
            pretrained_tag="timm_pretrained" if pretrained else "random_init",
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.backbone(x)
        if out.ndim != 2:
            raise RuntimeError(
                f"Appearance branch expected pooled 2D output, got shape {tuple(out.shape)}"
            )
        return out


class TinyCLIPImageBranch(nn.Module):
    """Semantic branch that loads TinyCLIP with a deterministic fallback chain."""

    _OPEN_CLIP_CHAIN = (
        {
            "model_name": "hf-hub:wkcn/TinyCLIP-ViT-45M-32-Text-21M-LAION400M",
            "pretrained_tag": None,
        },
        {
            "model_name": "TinyCLIP-ViT-40M-32-Text-19M",
            "pretrained_tag": "laion400m_e32",
        },
    )
    _TIMM_TINYCLIP_CHAIN = (
        "vit_medium_patch32_clip_224.tinyclip_laion400m",
    )
    _LAST_RESORT_OPEN_CLIP = ("ViT-B-32", "openai")

    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.provider = ""
        self.model = None
        self.loaded_backbone: LoadedBackboneInfo | None = None
        last_error = self._try_load_open_clip(pretrained=pretrained)
        if self.model is None:
            last_error = self._try_load_timm_tinyclip(pretrained=pretrained) or last_error
        if self.model is None:
            last_error = self._try_load_open_clip_last_resort(pretrained=pretrained) or last_error

        if self.model is None or self.loaded_backbone is None:
            raise RuntimeError(
                "Unable to load any TinyCLIP/OpenCLIP visual backbone"
            ) from last_error

        self.image_size = self._infer_image_size(self.model)

    def _try_load_open_clip(self, pretrained: bool) -> Exception | None:
        try:
            import open_clip
        except ImportError as exc:
            return exc

        last_error: Exception | None = None
        for candidate in self._OPEN_CLIP_CHAIN:
            model_name = candidate["model_name"]
            pretrained_tag = candidate["pretrained_tag"]
            try:
                if pretrained_tag is None:
                    model, _, _ = open_clip.create_model_and_transforms(model_name)
                else:
                    model, _, _ = open_clip.create_model_and_transforms(
                        model_name,
                        pretrained=pretrained_tag if pretrained else None,
                    )
            except Exception as exc:  # noqa: BLE001 - preserve fallback chain context
                last_error = exc
                logger.warning(
                    "TinyCLIP load failed for model='%s' pretrained='%s': %s",
                    model_name,
                    pretrained_tag or "hf-hub-default",
                    exc,
                )
                continue

            self.model = model
            self.provider = "open_clip"
            self.loaded_backbone = LoadedBackboneInfo(
                family="semantic",
                model_name=model_name,
                pretrained_tag=pretrained_tag if pretrained else "random_init",
            )
            self.output_dim = self._infer_open_clip_output_dim(model)
            logger.info(
                "TinyCLIP branch loaded model='%s' pretrained='%s' via open_clip output_dim=%s",
                model_name,
                pretrained_tag if pretrained_tag is not None and pretrained else "hf-hub-default",
                self.output_dim,
            )
            return None

        return last_error

    def _try_load_timm_tinyclip(self, pretrained: bool) -> Exception | None:
        try:
            import timm
        except ImportError as exc:
            return exc

        last_error: Exception | None = None
        for model_name in self._TIMM_TINYCLIP_CHAIN:
            try:
                model = timm.create_model(
                    model_name,
                    pretrained=pretrained,
                    num_classes=0,
                )
            except Exception as exc:  # noqa: BLE001 - preserve fallback chain context
                last_error = exc
                logger.warning(
                    "TinyCLIP-equivalent timm load failed for model='%s': %s",
                    model_name,
                    exc,
                )
                continue

            self.model = model
            self.provider = "timm"
            self.loaded_backbone = LoadedBackboneInfo(
                family="semantic",
                model_name=model_name,
                pretrained_tag="timm_pretrained" if pretrained else "random_init",
            )
            self.output_dim = self._infer_timm_output_dim(model)
            logger.info(
                "TinyCLIP branch loaded model='%s' via timm output_dim=%s",
                model_name,
                self.output_dim,
            )
            return None

        return last_error

    def _try_load_open_clip_last_resort(self, pretrained: bool) -> Exception | None:
        try:
            import open_clip
        except ImportError as exc:
            return exc

        model_name, pretrained_tag = self._LAST_RESORT_OPEN_CLIP
        try:
            model, _, _ = open_clip.create_model_and_transforms(
                model_name,
                pretrained=pretrained_tag if pretrained else None,
            )
        except Exception as exc:  # noqa: BLE001 - explicit last resort context
            logger.warning(
                "OpenCLIP last resort load failed for model='%s' pretrained='%s': %s",
                model_name,
                pretrained_tag,
                exc,
            )
            return exc

        self.model = model
        self.provider = "open_clip"
        self.loaded_backbone = LoadedBackboneInfo(
            family="semantic",
            model_name=model_name,
            pretrained_tag=pretrained_tag if pretrained else "random_init",
        )
        self.output_dim = self._infer_open_clip_output_dim(model)
        logger.info(
            "TinyCLIP branch loaded model='%s' pretrained='%s' via open_clip output_dim=%s",
            model_name,
            pretrained_tag if pretrained else "random_init",
            self.output_dim,
        )
        return None

    @staticmethod
    def _infer_open_clip_output_dim(model: nn.Module) -> int:
        visual = getattr(model, "visual", None)
        output_dim = getattr(visual, "output_dim", None)
        if isinstance(output_dim, int):
            return output_dim

        visual_proj = getattr(model, "visual_projection", None)
        if isinstance(visual_proj, torch.Tensor) and visual_proj.ndim == 2:
            return int(visual_proj.shape[-1])

        visual_proj = getattr(visual, "proj", None)
        if isinstance(visual_proj, torch.Tensor):
            if visual_proj.ndim == 1:
                return int(visual_proj.shape[0])
            if visual_proj.ndim == 2:
                return int(visual_proj.shape[-1])

        raise RuntimeError("Could not infer TinyCLIP visual output dimension")

    @staticmethod
    def _infer_timm_output_dim(model: nn.Module) -> int:
        output_dim = getattr(model, "num_features", None)
        if isinstance(output_dim, int):
            return output_dim
        raise RuntimeError("Could not infer timm TinyCLIP visual output dimension")

    @staticmethod
    def _infer_image_size(model: nn.Module) -> tuple[int, int]:
        pretrained_cfg = getattr(model, "pretrained_cfg", None)
        if isinstance(pretrained_cfg, dict):
            input_size = pretrained_cfg.get("input_size")
            if isinstance(input_size, tuple) and len(input_size) == 3:
                return (int(input_size[-2]), int(input_size[-1]))

        visual = getattr(model, "visual", None)
        image_size = getattr(visual, "image_size", None)
        if isinstance(image_size, int):
            return (image_size, image_size)
        if isinstance(image_size, tuple) and len(image_size) == 2:
            return (int(image_size[0]), int(image_size[1]))
        return (224, 224)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if tuple(x.shape[-2:]) != self.image_size:
            x = F.interpolate(
                x,
                size=self.image_size,
                mode="bilinear",
                align_corners=False,
            )
        if self.provider == "open_clip":
            features = self.model.encode_image(x, normalize=False)
        else:
            features = self.model(x)
        if features.ndim != 2:
            raise RuntimeError(
                f"TinyCLIP branch expected 2D image features, got shape {tuple(features.shape)}"
            )
        return features


class CLIPSENet(nn.Module):
    """CLIP-SENet with a CNN appearance branch and a CLIP semantic branch."""

    def __init__(
        self,
        num_classes: int,
        embed_dim: int = 2048,
        afem_groups: int = 32,
        feat_dim_appearance: int = 2048,
        feat_dim_semantic: int = 512,
        dropout: float = 0.0,
        appearance_pretrained: bool = True,
        semantic_pretrained: bool = True,
        residual_mode: str = "grouped_identity",
    ):
        super().__init__()
        self.num_classes = num_classes
        self.embed_dim = embed_dim

        self.appearance_branch = ResNet101IBNBranch(pretrained=appearance_pretrained)
        self.semantic_branch = TinyCLIPImageBranch(pretrained=semantic_pretrained)

        detected_app_dim = self.appearance_branch.output_dim
        detected_sem_dim = self.semantic_branch.output_dim
        if feat_dim_appearance != detected_app_dim:
            logger.warning(
                "Requested feat_dim_appearance=%s but backbone reports %s. Using detected dim.",
                feat_dim_appearance,
                detected_app_dim,
            )
        if feat_dim_semantic != detected_sem_dim:
            logger.warning(
                "Requested feat_dim_semantic=%s but backbone reports %s. Using detected dim.",
                feat_dim_semantic,
                detected_sem_dim,
            )

        self.feat_dim_appearance = detected_app_dim
        self.feat_dim_semantic = detected_sem_dim
        self.fusion_fc = nn.Linear(
            self.feat_dim_appearance + self.feat_dim_semantic,
            embed_dim,
            bias=False,
        )
        self.afem = AFEMBlock(
            in_dim=embed_dim,
            out_dim=embed_dim,
            num_groups=afem_groups,
            residual_mode=residual_mode,
        )
        self.bnneck = nn.BatchNorm1d(embed_dim)
        self.bnneck.bias.requires_grad_(False)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.classifier = nn.Linear(embed_dim, num_classes, bias=False)

        nn.init.kaiming_normal_(self.fusion_fc.weight, mode="fan_out")
        nn.init.normal_(self.classifier.weight, std=0.001)

        self.loaded_resnext_model = self.appearance_branch.loaded_backbone.model_name
        self.loaded_tinyclip_model = self.semantic_branch.loaded_backbone.model_name

    def forward(self, x: torch.Tensor):
        f_app = self.appearance_branch(x)
        f_sem = self.semantic_branch(x)
        t_u = self.fusion_fc(torch.cat([f_app, f_sem], dim=1))
        t_s_prime = self.afem(t_u)
        t = t_u + t_s_prime
        t_bn = self.bnneck(t)
        t_bn_normalized = F.normalize(t_bn, p=2, dim=1)

        if self.training:
            logits = self.classifier(self.dropout(t_bn))
            return t_bn_normalized, logits

        return t_bn_normalized


def build_clip_senet(num_classes: int, **kwargs) -> CLIPSENet:
    """Build a CLIP-SENet model for M1 architecture validation."""

    return CLIPSENet(num_classes=num_classes, **kwargs)


import json
from pathlib import Path

DEFAULT_NUM_CLASSES = 666
CHECKPOINT_CANDIDATE_NAMES = ("best.pth", "last.pth", "vehicle_clip_senet_cityflow_ft.pth")
PREFERRED_EXACT = Path("/kaggle/input/notebooks/yahiaakhalafallah/13f-clip-senet-cityflow-finetune/checkpoints/best.pth")
PREFERRED_SOURCE_SLUG = "13f-clip-senet-cityflow-finetune"


def torch_load(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def discover_clip_senet_checkpoint() -> Path:
    if PREFERRED_EXACT.is_file():
        print(f"Using preferred 13f checkpoint: {PREFERRED_EXACT}")
        return PREFERRED_EXACT

    candidates = []
    for root in sorted(Path("/kaggle/input").rglob("*")):
        if not root.is_dir():
            continue
        for name in CHECKPOINT_CANDIDATE_NAMES:
            candidate = root / name
            if candidate.is_file():
                score = 0
                candidate_str = str(candidate).lower()
                if PREFERRED_SOURCE_SLUG in candidate_str:
                    score += 100
                if "checkpoints" in candidate.parts:
                    score += 10
                if name == "best.pth":
                    score += 5
                candidates.append((score, candidate))
    if not candidates:
        raise FileNotFoundError(
            "Could not find 13f CLIP-SENet checkpoint under /kaggle/input. "
            "Expected kernel source yahiaakhalafallah/13f-clip-senet-cityflow-finetune."
        )
    candidates.sort(key=lambda item: (-item[0], str(item[1])))
    print("Checkpoint candidates:")
    for score, path in candidates[:10]:
        print(f"  score={score:3d}  {path}")
    return candidates[0][1]


CHECKPOINT_PATH = discover_clip_senet_checkpoint()
payload = torch_load(CHECKPOINT_PATH)
if isinstance(payload, dict) and "model_state" in payload:
    state_dict = payload["model_state"]
    checkpoint_kind = "payload:model_state"
elif isinstance(payload, dict) and "model" in payload and isinstance(payload["model"], dict):
    state_dict = payload["model"]
    checkpoint_kind = "payload:model"
elif isinstance(payload, dict) and payload and all(hasattr(value, "shape") for value in payload.values()):
    state_dict = payload
    checkpoint_kind = "state_dict"
else:
    raise TypeError(f"Unsupported checkpoint format at {CHECKPOINT_PATH}: {type(payload).__name__}")

classifier_weight = state_dict.get("classifier.weight")
inferred_num_classes = int(classifier_weight.shape[0]) if classifier_weight is not None else DEFAULT_NUM_CLASSES
model = build_clip_senet(num_classes=inferred_num_classes).to(DEVICE)
missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
if missing_keys or unexpected_keys:
    print("Missing keys:", missing_keys)
    print("Unexpected keys:", unexpected_keys)
    raise RuntimeError("13f checkpoint load was not strict; aborting feature extraction setup")
model.classifier = torch.nn.Identity()
model.eval()
FEATURE_DIM = int(getattr(model, "embed_dim", 2048))

RUN_INFO = {
    "device": DEVICE,
    "checkpoint_path": str(CHECKPOINT_PATH),
    "checkpoint_kind": checkpoint_kind,
    "num_classes": inferred_num_classes,
    "feature_dim": FEATURE_DIM,
    "loaded_tinyclip_model": getattr(model, "loaded_tinyclip_model", None),
    "loaded_appearance_model": getattr(model, "loaded_resnext_model", None),
    "source_kernel": "yahiaakhalafallah/13f-clip-senet-cityflow-finetune",
}
print(json.dumps(RUN_INFO, indent=2))

In [ ]:
from PIL import Image
import torchvision.transforms as T
from tqdm.auto import tqdm

from src.core.io_utils import load_embeddings, load_tracklets_by_camera
from src.stage2_features.embeddings import l2_normalize
from src.stage2_features.pca_whitening import PCAWhitener

PREV_SLUG = "mtmc-10b-stage-3-faiss-indexing"


def find_input_dir(slug: str, hints=()) -> Path:
    direct = INPUT_ROOT / slug
    if direct.exists():
        return direct
    for path in INPUT_ROOT.iterdir():
        name = path.name.lower()
        if slug.lower() in name or any(hint.lower() in name for hint in hints):
            return path
    return direct


PREV_INPUT = find_input_dir(PREV_SLUG, hints=("10b", "stage-3", "faiss"))
checkpoint = PREV_INPUT / "checkpoint.tar.gz"
if not checkpoint.exists():
    print(f"checkpoint.tar.gz not found at {checkpoint}; trying Kaggle API fallback")
    dl_dir = Path("/tmp/kaggle_10b_download")
    dl_dir.mkdir(parents=True, exist_ok=True)
    for candidate in [
        "yahiaakhalafallah/mtmc-10b-stage-3-faiss-indexing",
        "gumfreddy/mtmc-10b-stage-3-faiss-indexing",
    ]:
        result = subprocess.run(
            ["kaggle", "kernels", "output", candidate, "--file", "checkpoint.tar.gz", "-p", str(dl_dir)],
            capture_output=True,
            text=True,
        )
        print(result.stdout)
        print(result.stderr)
        checkpoint = dl_dir / "checkpoint.tar.gz"
        if checkpoint.exists():
            break

assert checkpoint.exists(), f"checkpoint.tar.gz not found for {PREV_SLUG}"

EXTRACT_DIR = Path("/tmp/pipeline_run")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Extracting {checkpoint} ({checkpoint.stat().st_size / 1024**2:.1f} MB)")
with tarfile.open(str(checkpoint), "r:gz") as tar:
    tar.extractall(str(EXTRACT_DIR))

with open(EXTRACT_DIR / "run_metadata.json", encoding="utf-8") as handle:
    meta = json.load(handle)

RUN_NAME = meta["run_name"]
DATA_OUT = EXTRACT_DIR
RUN_DIR = EXTRACT_DIR / RUN_NAME
STAGE1_DIR = RUN_DIR / "stage1"
STAGE2_DIR = RUN_DIR / "stage2"
STAGE3_DIR = RUN_DIR / "stage3"
PRIMARY_EMBEDDINGS_PATH = STAGE2_DIR / "embeddings.npy"
TERTIARY_DINOV2_PATH = STAGE2_DIR / "embeddings_tertiary.npy"

assert PRIMARY_EMBEDDINGS_PATH.exists(), PRIMARY_EMBEDDINGS_PATH
assert TERTIARY_DINOV2_PATH.exists(), (
    f"Missing 10c v15 DINOv2 tertiary embeddings at {TERTIARY_DINOV2_PATH}. "
    "Do not run this sweep without the TransReID+DINOv2 control input."
)

primary_embeddings, embedding_index = load_embeddings(STAGE2_DIR)
tracklets_by_camera = load_tracklets_by_camera(STAGE1_DIR)
tertiary_embeddings = np.load(TERTIARY_DINOV2_PATH)

assert primary_embeddings.shape[0] == len(embedding_index)
assert tertiary_embeddings.shape[0] == len(embedding_index)

repo_gt = PROJECT / "data" / "raw" / "cityflowv2"
dataset_gt = Path("/kaggle/input/data-aicity-2023-track-2")
if any((repo_gt / cam / "gt" / "gt.txt").exists() for cam in ["S01_c001", "S01_c002", "S01_c003", "S02_c006", "S02_c007", "S02_c008"]):
    GT_DIR = str(repo_gt)
elif dataset_gt.exists():
    GT_DIR = str(dataset_gt)
else:
    GT_DIR = ""

CAM_RE = re.compile(r"^S\d{2}_c\d{3}$")
VIDEO_EXTS = {".mp4", ".avi", ".mkv", ".mov"}
OUTPUT_DIR = WORK_DIR / "13f_clip_senet_features"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def looks_like_cityflow_root(path: Path) -> bool:
    if not path.is_dir():
        return False
    return any((path / split).is_dir() for split in ("validation", "train", "test"))


def discover_cityflow_root() -> Path:
    explicit = [
        Path("/kaggle/input/data-aicity-2023-track-2"),
        Path("/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2"),
    ]
    for candidate in explicit:
        if looks_like_cityflow_root(candidate):
            return candidate
        if candidate.exists():
            for nested in candidate.rglob("*"):
                if looks_like_cityflow_root(nested):
                    return nested
    scored = []
    for candidate in INPUT_ROOT.rglob("*"):
        if looks_like_cityflow_root(candidate):
            score = 0
            text = str(candidate).lower()
            if "data-aicity-2023-track-2" in text:
                score += 100
            if "cityflow" in text or "aic" in text:
                score += 20
            scored.append((score, candidate))
    if not scored:
        raise FileNotFoundError("CityFlowV2 root not found under /kaggle/input")
    scored.sort(key=lambda item: (-item[0], str(item[1])))
    return scored[0][1]


def camera_id_from_scene_cam(scene_dir: Path, cam_dir: Path) -> str:
    return f"{scene_dir.name}_{cam_dir.name}"


def discover_cityflow_videos(cityflow_root: Path):
    videos = {}
    split_by_camera = {}
    for split_name in ("validation", "train", "test"):
        split_dir = cityflow_root / split_name
        if not split_dir.is_dir():
            continue
        for scene_dir in sorted(split_dir.iterdir()):
            if not scene_dir.is_dir():
                continue
            for cam_dir in sorted(scene_dir.iterdir()):
                if not cam_dir.is_dir():
                    continue
                cam_id = camera_id_from_scene_cam(scene_dir, cam_dir)
                cam_videos = [p for p in cam_dir.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTS]
                if not cam_videos:
                    cam_videos = [p for p in cam_dir.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTS]
                if cam_videos:
                    videos[cam_id] = sorted(cam_videos)[0]
                    split_by_camera[cam_id] = split_name
    if not videos:
        raise FileNotFoundError(f"No camera videos found below {cityflow_root}")
    return videos, split_by_camera


IMAGE_SIZE = (320, 320)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
BATCH_SIZE = 64
transform = T.Compose([
    T.Resize(IMAGE_SIZE, interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def apply_cityflow_stage0_preprocess(frame_bgr: np.ndarray) -> np.ndarray:
    lab = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    l_channel = clahe.apply(l_channel)
    return cv2.cvtColor(cv2.merge((l_channel, a_channel, b_channel)), cv2.COLOR_LAB2BGR)


def crop_xyxy(frame_bgr: np.ndarray, bbox):
    height, width = frame_bgr.shape[:2]
    x1, y1, x2, y2 = [float(value) for value in bbox]
    x1 = int(max(0, min(width - 1, round(x1))))
    y1 = int(max(0, min(height - 1, round(y1))))
    x2 = int(max(0, min(width, round(x2))))
    y2 = int(max(0, min(height, round(y2))))
    if x2 <= x1 or y2 <= y1:
        return None
    crop_bgr = frame_bgr[y1:y2, x1:x2]
    if crop_bgr.size == 0:
        return None
    return Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))


def read_frame(cap: cv2.VideoCapture, frame_id: int):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_id))
    ok, frame = cap.read()
    if not ok or frame is None:
        return None
    return frame


def feature_batch(images):
    if not images:
        return np.empty((0, FEATURE_DIM), dtype=np.float32)
    tensor = torch.stack([transform(image) for image in images], dim=0).to(DEVICE, non_blocking=True)
    with torch.no_grad():
        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            features = model(tensor)
        features = torch.nn.functional.normalize(features.float(), p=2, dim=1)
    return features.cpu().numpy().astype(np.float32)


CITYFLOW_ROOT = discover_cityflow_root()
CAMERA_VIDEOS, CAMERA_SPLITS = discover_cityflow_videos(CITYFLOW_ROOT)
TRACKLET_FILES = sorted(STAGE1_DIR.glob("tracklets_*.json"))

print(f"Run name: {RUN_NAME}")
print(f"Stage 2 primary: {primary_embeddings.shape} -> {PRIMARY_EMBEDDINGS_PATH}")
print(f"Stage 2 DINOv2 tertiary: {tertiary_embeddings.shape} -> {TERTIARY_DINOV2_PATH}")
print(f"Embedding index rows: {len(embedding_index)}")
print(f"Stage 1 cameras: {sorted(tracklets_by_camera)}")
print(f"CITYFLOW_ROOT: {CITYFLOW_ROOT}")
print(f"GT dir: {GT_DIR or 'not found'}")

global_index = []
feature_manifest = {
    "run_info": RUN_INFO,
    "cityflow_root": str(CITYFLOW_ROOT),
    "stage1_dir": str(STAGE1_DIR),
    "output_dir": str(OUTPUT_DIR),
    "image_size": list(IMAGE_SIZE),
    "normalization": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD},
    "per_camera": {},
    "index": global_index,
}

total_detections = 0
skipped_zero_conf = 0
skipped_missing_video = 0
skipped_missing_frame = 0
skipped_bad_crop = 0
feature_dim_seen = None

for tracklet_file in TRACKLET_FILES:
    camera_id = tracklet_file.stem.replace("tracklets_", "")
    if camera_id not in tracklets_by_camera:
        continue
    video_path = CAMERA_VIDEOS.get(camera_id)
    if video_path is None:
        print(f"SKIP {camera_id}: no CityFlowV2 video found")
        skipped_missing_video += 1
        continue

    tracklets = json.loads(tracklet_file.read_text())
    records_by_frame = defaultdict(list)
    det_counter = 0
    for tracklet in tracklets:
        track_id = int(tracklet["track_id"])
        for frame in tracklet.get("frames", []):
            confidence = float(frame.get("confidence", 1.0))
            if confidence <= 0.0:
                skipped_zero_conf += 1
                continue
            record = {
                "det_id": det_counter,
                "track_id": track_id,
                "frame_id": int(frame["frame_id"]),
                "bbox": [float(value) for value in frame["bbox"]],
                "confidence": confidence,
            }
            records_by_frame[record["frame_id"]].append(record)
            det_counter += 1

    embeddings = []
    frame_ids = []
    bbox_xyxy = []
    track_ids = []
    det_ids = []
    confidences = []
    pending_images = []
    pending_records = []

    def flush_batch():
        nonlocal_feature_count = len(pending_images)
        if nonlocal_feature_count == 0:
            return
        batch_embeddings = feature_batch(pending_images)
        embeddings.append(batch_embeddings)
        for local_record, embedding in zip(pending_records, batch_embeddings):
            local_index = len(frame_ids)
            global_feature_index = total_detections + local_index
            frame_ids.append(local_record["frame_id"])
            bbox_xyxy.append(local_record["bbox"])
            track_ids.append(local_record["track_id"])
            det_ids.append(local_record["det_id"])
            confidences.append(local_record["confidence"])
            global_index.append({
                "camera_id": camera_id,
                "frame_id": int(local_record["frame_id"]),
                "det_id": int(local_record["det_id"]),
                "track_id": int(local_record["track_id"]),
                "feature_index": int(global_feature_index),
                "local_index": int(local_index),
                "bbox_xyxy": [float(value) for value in local_record["bbox"]],
            })
        pending_images.clear()
        pending_records.clear()

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise IOError(f"Cannot open video for {camera_id}: {video_path}")

    try:
        progress_total = sum(len(items) for items in records_by_frame.values())
        pbar = tqdm(total=progress_total, desc=f"{camera_id}")
        for frame_id in sorted(records_by_frame):
            raw_frame = read_frame(cap, frame_id)
            if raw_frame is None:
                skipped_missing_frame += len(records_by_frame[frame_id])
                pbar.update(len(records_by_frame[frame_id]))
                continue
            frame_bgr = apply_cityflow_stage0_preprocess(raw_frame)
            for record in records_by_frame[frame_id]:
                crop = crop_xyxy(frame_bgr, record["bbox"])
                if crop is None:
                    skipped_bad_crop += 1
                    pbar.update(1)
                    continue
                pending_images.append(crop)
                pending_records.append(record)
                if len(pending_images) >= BATCH_SIZE:
                    flush_batch()
                pbar.update(1)
        flush_batch()
        pbar.close()
    finally:
        cap.release()

    embeddings_arr = np.concatenate(embeddings, axis=0).astype(np.float32) if embeddings else np.empty((0, FEATURE_DIM), dtype=np.float32)
    frame_ids_arr = np.asarray(frame_ids, dtype=np.int64)
    bbox_arr = np.asarray(bbox_xyxy, dtype=np.float32).reshape(-1, 4)
    track_ids_arr = np.asarray(track_ids, dtype=np.int64)
    det_ids_arr = np.asarray(det_ids, dtype=np.int64)
    confidences_arr = np.asarray(confidences, dtype=np.float32)

    if embeddings_arr.size:
        norms = np.linalg.norm(embeddings_arr, axis=1)
        print(f"{camera_id}: norm min/mean/max = {norms.min():.6f}/{norms.mean():.6f}/{norms.max():.6f}")
        feature_dim_seen = int(embeddings_arr.shape[1])

    output_path = OUTPUT_DIR / f"{camera_id}.npz"
    np.savez_compressed(
        output_path,
        embeddings=embeddings_arr,
        frame_ids=frame_ids_arr,
        bbox_xyxy=bbox_arr,
        track_ids=track_ids_arr,
        det_ids=det_ids_arr,
        confidences=confidences_arr,
    )
    file_size_mb = output_path.stat().st_size / 1024**2
    feature_manifest["per_camera"][camera_id] = {
        "file": str(output_path),
        "split": CAMERA_SPLITS.get(camera_id),
        "video": str(video_path),
        "tracklet_file": str(tracklet_file),
        "count": int(embeddings_arr.shape[0]),
        "feature_dim": int(embeddings_arr.shape[1]) if embeddings_arr.ndim == 2 else FEATURE_DIM,
        "offset": int(total_detections),
        "file_size_mb": round(file_size_mb, 3),
    }
    total_detections += int(embeddings_arr.shape[0])
    print(f"Saved {camera_id}: {embeddings_arr.shape} -> {output_path} ({file_size_mb:.1f} MB)")

feature_manifest["summary"] = {
    "total_detections": int(total_detections),
    "total_cameras": int(len(feature_manifest["per_camera"])),
    "feature_dim": int(feature_dim_seen or FEATURE_DIM),
    "skipped_zero_conf_interpolated": int(skipped_zero_conf),
    "skipped_missing_video_cameras": int(skipped_missing_video),
    "skipped_missing_frames": int(skipped_missing_frame),
    "skipped_bad_crops": int(skipped_bad_crop),
}

camera_npz = {}
for camera_id in sorted(tracklets_by_camera):
    npz_path = OUTPUT_DIR / f"{camera_id}.npz"
    assert npz_path.exists(), f"Missing 13f CLIP-SENet NPZ for {camera_id}: {npz_path}"
    camera_npz[camera_id] = np.load(npz_path)

source_keys_by_camera = {}
for row in embedding_index:
    key = (row["camera_id"], int(row["track_id"]))
    source_keys_by_camera.setdefault(row["camera_id"], set()).add(key)


def clip_tracklet_keys(camera_id: str, data: np.lib.npyio.NpzFile) -> set[tuple[str, int]]:
    track_ids = data["track_ids"].astype(np.int64)
    return {(camera_id, int(track_id)) for track_id in np.unique(track_ids)}


alignment_report = {}
alignment_problems = {}
for camera_id, data in camera_npz.items():
    source_keys = source_keys_by_camera.get(camera_id, set())
    clip_keys = clip_tracklet_keys(camera_id, data)
    missing_from_clip = sorted(source_keys - clip_keys)
    extra_in_clip = sorted(clip_keys - source_keys)
    alignment_report[camera_id] = {
        "detections": int(data["frame_ids"].shape[0]),
        "source_tracklets": int(len(source_keys)),
        "clip_tracklets": int(len(clip_keys)),
        "feature_dim": int(data["embeddings"].shape[1]),
        "missing_from_clip": missing_from_clip[:10],
        "extra_in_clip": extra_in_clip[:10],
    }
    if missing_from_clip or extra_in_clip:
        alignment_problems[camera_id] = alignment_report[camera_id]

if alignment_problems:
    print(json.dumps(alignment_problems, indent=2))
    raise RuntimeError("13f tracklet alignment failed; CLIP-SENet and Stage 4 tracklet key sets differ")

clip_by_camera_track = {}
for camera_id, data in camera_npz.items():
    embeddings = l2_normalize(data["embeddings"].astype(np.float32))
    track_ids = data["track_ids"].astype(np.int64)
    for track_id in np.unique(track_ids):
        mask = track_ids == track_id
        pooled = embeddings[mask].mean(axis=0)
        norm = np.linalg.norm(pooled)
        if norm <= 1e-8:
            raise RuntimeError(f"Zero pooled 13f CLIP-SENet embedding for {camera_id} track {int(track_id)}")
        clip_by_camera_track[(camera_id, int(track_id))] = (pooled / norm).astype(np.float32)

source_key_set = {(row["camera_id"], int(row["track_id"])) for row in embedding_index}
extra_clip_keys = sorted(set(clip_by_camera_track) - source_key_set)
if extra_clip_keys:
    print(f"Skipping {len(extra_clip_keys)} 13f CLIP-SENet tracklets not present in Stage 4 source rows: {extra_clip_keys[:10]}")

clip_rows = []
missing = []
for row in embedding_index:
    key = (row["camera_id"], int(row["track_id"]))
    value = clip_by_camera_track.get(key)
    if value is None:
        missing.append(key)
    else:
        clip_rows.append(value)

if missing:
    print(f"Missing {len(missing)} 13f CLIP-SENet tracklets required by Stage 4 source rows: {missing[:10]}")
    raise RuntimeError(f"Missing 13f CLIP-SENet tracklet embeddings for {len(missing)} Stage 2 rows")

clip_raw = np.stack(clip_rows, axis=0).astype(np.float32)
clip_raw = l2_normalize(clip_raw)
raw_path = STAGE2_DIR / "embeddings_13f_clip_senet_raw_tracklet.npy"
np.save(raw_path, clip_raw)

pca_components = min(384, clip_raw.shape[0], clip_raw.shape[1])
if pca_components < 50:
    raise RuntimeError(f"Too few 13f CLIP-SENet tracklets for stable PCA: {clip_raw.shape}")

whitener = PCAWhitener(n_components=pca_components)
clip_pca = whitener.fit_transform(clip_raw)
clip_pca = l2_normalize(clip_pca)

CLIP_TRACKLET_EMBEDDINGS_PATH = WORK_DIR / "13f_features.npy"
CLIP_STAGE2_PATH = STAGE2_DIR / "embeddings_13f_clip_senet.npy"
CLIP_PCA_PATH = STAGE2_DIR / "pca_13f_clip_senet_tracklet.pkl"
np.save(CLIP_TRACKLET_EMBEDDINGS_PATH, clip_pca.astype(np.float32))
np.save(CLIP_STAGE2_PATH, clip_pca.astype(np.float32))
whitener.save(CLIP_PCA_PATH)

feature_manifest["tracklet_features"] = {
    "raw_path": str(raw_path),
    "pca_path": str(CLIP_TRACKLET_EMBEDDINGS_PATH),
    "stage2_copy": str(CLIP_STAGE2_PATH),
    "pca_model": str(CLIP_PCA_PATH),
    "shape": list(clip_pca.shape),
    "granularity": "tracklet",
}
feature_manifest["alignment_report"] = alignment_report
manifest_path = WORK_DIR / "13h_13f_feature_manifest.json"
manifest_path.write_text(json.dumps(feature_manifest, indent=2), encoding="utf-8")

assert clip_pca.shape[0] == primary_embeddings.shape[0] == tertiary_embeddings.shape[0]
print("13f tracklet key alignment verified against Stage 4 embedding index")
print(f"13f raw tracklet embeddings: {clip_raw.shape} -> {raw_path}")
print(f"13f PCA/L2 tracklet embeddings: {clip_pca.shape} -> {CLIP_TRACKLET_EMBEDDINGS_PATH}")
print(f"Feature manifest: {manifest_path}")
print(json.dumps(feature_manifest["summary"], indent=2))

In [ ]:
AQE_K = 3
SIM_THRESH = 0.50
SOLVER = "cc"
ALGORITHM = "conflict_free_cc"
LOUVAIN_RES = 0.70

APPEARANCE_WEIGHT = 0.70
HSV_WEIGHT = 0.0
ST_WEIGHT = round(1.0 - APPEARANCE_WEIGHT - HSV_WEIGHT, 4)

BRIDGE_PRUNE = 0.0
MAX_COMP_SIZE = 12
FIC_REG = 0.50
GALLERY_THRESH = 0.48
ORPHAN_MATCH_THRESH = 0.38

INTRA_MERGE = True
INTRA_MERGE_THRESH = 0.80
INTRA_MERGE_GAP = 30

BASELINE_DINOV2_WEIGHT = 0.60
WEIGHT_SWEEP = [0.0, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 1.0]

MULTI_QUERY_WEIGHT = 0.00
CAMERA_BIAS = False
CAMERA_BIAS_ITERS = 2
ZONE_MODEL = False
ZONE_BONUS = 0.06
ZONE_PENALTY = 0.04
HIERARCHICAL = False
HIER_CENTROID_TH = 0.45
HIER_MERGE_TH = 0.45
HIER_ORPHAN_TH = 0.40
RERANKING = False
RERANKING_K1 = 20
RERANKING_K2 = 6
RERANKING_LAMBDA = 0.5
CAMERA_PAIR_NORM = False
AFLINK_ENABLED = False
AFLINK_MAX_TIME_GAP_FRAMES = 150
AFLINK_MAX_SPATIAL_GAP_PX = 150.0
AFLINK_MIN_DIRECTION_COS = 0.85
AFLINK_MIN_VELOCITY_RATIO = 0.5
AFLINK_VELOCITY_WINDOW = 5
MTMC_ONLY = False


def secondary_embedding_overrides(weight: float) -> list[str]:
    if weight <= 0.0:
        return [
            "--override", "stage4.association.secondary_embeddings.path=",
            "--override", "stage4.association.secondary_embeddings.weight=0.0",
        ]
    return [
        "--override", f"stage4.association.secondary_embeddings.path={CLIP_TRACKLET_EMBEDDINGS_PATH}",
        "--override", f"stage4.association.secondary_embeddings.weight={weight}",
    ]


def tertiary_embedding_overrides(weight: float) -> list[str]:
    if weight <= 0.0:
        return [
            "--override", "stage4.association.tertiary_embeddings.path=",
            "--override", "stage4.association.tertiary_embeddings.weight=0.0",
        ]
    return [
        "--override", f"stage4.association.tertiary_embeddings.path={TERTIARY_DINOV2_PATH}",
        "--override", f"stage4.association.tertiary_embeddings.weight={weight}",
    ]


def ensure_upstream_links(run_name: str) -> Path:
    run_dir = DATA_OUT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    for stage_name in ("stage1", "stage2", "stage3"):
        src = RUN_DIR / stage_name
        dst = run_dir / stage_name
        if dst.exists():
            continue
        try:
            dst.symlink_to(src, target_is_directory=True)
        except OSError:
            shutil.copytree(src, dst, dirs_exist_ok=True)
    return run_dir


def load_metrics(report_path: Path) -> dict:
    if not report_path.exists():
        return {}
    payload = json.loads(report_path.read_text(encoding="utf-8"))
    details = payload.get("details", {}) or {}
    error_analysis = details.get("error_analysis", {}) or {}
    return {
        "mtmc_idf1": payload.get("mtmc_idf1", details.get("mtmc_idf1", payload.get("idf1"))),
        "idf1": payload.get("idf1"),
        "mota": payload.get("mota"),
        "hota": payload.get("hota"),
        "id_switches": payload.get("id_switches"),
        "conflations": error_analysis.get("conflated_pred"),
        "fragmentations": error_analysis.get("fragmented_gt"),
        "num_pred_ids": payload.get("num_pred_ids", error_analysis.get("total_pred")),
    }


def count_clusters(run_name: str) -> int | None:
    path = DATA_OUT / run_name / "stage4" / "global_trajectories.json"
    if not path.exists():
        return None
    return len(json.loads(path.read_text(encoding="utf-8")))


def build_cmd(run_name: str, w_cs_ft: float) -> tuple[list[str], float, float, float]:
    w_dino = round((1.0 - w_cs_ft) * BASELINE_DINOV2_WEIGHT, 6)
    w_primary = round(1.0 - w_cs_ft - w_dino, 6)
    if w_primary < -1e-8:
        raise ValueError(f"Invalid weights: w_primary={w_primary}, w_dino={w_dino}, w_cs_ft={w_cs_ft}")
    cmd = [
        sys.executable, "scripts/run_pipeline.py",
        "--config", "configs/default.yaml",
        "--dataset-config", "configs/datasets/cityflowv2.yaml",
        "--stages", "4,5",
        "--override", f"project.run_name={run_name}",
        "--override", f"project.output_dir={DATA_OUT}",
        "--override", f"stage4.association.query_expansion.k={AQE_K}",
        "--override", "stage4.association.query_expansion.alpha=5.0",
        "--override", "stage4.association.query_expansion.dba=false",
        "--override", f"stage4.association.graph.similarity_threshold={SIM_THRESH}",
        "--override", f"stage4.association.solver={SOLVER}",
        "--override", f"stage4.association.graph.algorithm={ALGORITHM}",
        "--override", f"stage4.association.graph.louvain_resolution={LOUVAIN_RES}",
        "--override", f"stage4.association.graph.bridge_prune_margin={BRIDGE_PRUNE}",
        "--override", f"stage4.association.graph.max_component_size={MAX_COMP_SIZE}",
        "--override", f"stage4.association.weights.vehicle.appearance={APPEARANCE_WEIGHT}",
        "--override", f"stage4.association.weights.vehicle.hsv={HSV_WEIGHT}",
        "--override", f"stage4.association.weights.vehicle.spatiotemporal={ST_WEIGHT}",
        "--override", "stage4.association.mutual_nn.top_k_per_query=20",
        "--override", "stage4.association.fic.enabled=true",
        "--override", f"stage4.association.fic.regularisation={FIC_REG}",
        "--override", f"stage4.association.reranking.enabled={str(RERANKING).lower()}",
        "--override", f"stage4.association.reranking.k1={RERANKING_K1}",
        "--override", f"stage4.association.reranking.k2={RERANKING_K2}",
        "--override", f"stage4.association.reranking.lambda_value={RERANKING_LAMBDA}",
        "--override", f"stage4.association.camera_pair_norm.enabled={str(CAMERA_PAIR_NORM).lower()}",
        "--override", "stage4.association.fac.enabled=false",
        "--override", f"stage4.association.multi_query.enabled={str(MULTI_QUERY_WEIGHT > 0.0).lower()}",
        "--override", f"stage4.association.multi_query.weight={MULTI_QUERY_WEIGHT}",
        "--override", f"stage4.association.aflink.enabled={str(AFLINK_ENABLED).lower()}",
        "--override", f"stage4.association.aflink.max_time_gap_frames={AFLINK_MAX_TIME_GAP_FRAMES}",
        "--override", f"stage4.association.aflink.max_spatial_gap_px={AFLINK_MAX_SPATIAL_GAP_PX}",
        "--override", f"stage4.association.aflink.min_direction_cos={AFLINK_MIN_DIRECTION_COS}",
        "--override", f"stage4.association.aflink.min_velocity_ratio={AFLINK_MIN_VELOCITY_RATIO}",
        "--override", f"stage4.association.aflink.velocity_window={AFLINK_VELOCITY_WINDOW}",
        *secondary_embedding_overrides(w_cs_ft),
        *tertiary_embedding_overrides(w_dino),
        "--override", f"stage4.association.camera_bias.enabled={str(CAMERA_BIAS).lower()}",
        "--override", f"stage4.association.camera_bias.iterations={CAMERA_BIAS_ITERS}",
        "--override", f"stage4.association.zone_model.enabled={str(ZONE_MODEL).lower()}",
        "--override", "stage4.association.zone_model.zone_data_path=configs/datasets/cityflowv2_zones.json",
        "--override", f"stage4.association.zone_model.bonus={ZONE_BONUS}",
        "--override", f"stage4.association.zone_model.penalty={ZONE_PENALTY}",
        "--override", f"stage4.association.hierarchical.enabled={str(HIERARCHICAL).lower()}",
        "--override", f"stage4.association.hierarchical.centroid_threshold={HIER_CENTROID_TH}",
        "--override", f"stage4.association.hierarchical.merge_threshold={HIER_MERGE_TH}",
        "--override", f"stage4.association.hierarchical.orphan_threshold={HIER_ORPHAN_TH}",
        "--override", "stage4.association.hierarchical.max_merge_size=12",
        "--override", f"stage4.association.intra_camera_merge.enabled={str(INTRA_MERGE).lower()}",
        "--override", f"stage4.association.intra_camera_merge.threshold={INTRA_MERGE_THRESH}",
        "--override", f"stage4.association.intra_camera_merge.max_time_gap={INTRA_MERGE_GAP}",
        "--override", "stage4.association.gallery_expansion.enabled=true",
        "--override", f"stage4.association.gallery_expansion.threshold={GALLERY_THRESH}",
        "--override", f"stage4.association.gallery_expansion.orphan_match_threshold={ORPHAN_MATCH_THRESH}",
        "--override", "stage4.association.weights.length_weight_power=0.3",
        "--override", "stage4.association.temporal_overlap.enabled=true",
        "--override", "stage4.association.temporal_overlap.bonus=0.05",
        "--override", "stage4.association.temporal_overlap.max_mean_time=5.0",
        "--override", f"stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}",
        "--override", "stage5.stationary_filter.enabled=true",
        "--override", "stage5.stationary_filter.min_displacement_px=150",
        "--override", "stage5.stationary_filter.max_mean_velocity_px=2.0",
        "--override", "stage5.min_submission_confidence=0.15",
        "--override", "stage5.cross_id_nms_iou=0.40",
        "--override", "stage5.min_trajectory_confidence=0.30",
        "--override", "stage5.min_trajectory_frames=40",
        "--override", "stage5.track_edge_trim.enabled=false",
        "--override", "stage5.track_smoothing.enabled=false",
        "--override", "stage5.gt_frame_clip=true",
        "--override", "stage5.gt_zone_filter=true",
    ]
    if GT_DIR:
        cmd += ["--override", f"stage5.ground_truth_dir={GT_DIR}"]
    return cmd, w_primary, w_dino, w_cs_ft


print("Locked config:")
print(f"  AQE_K={AQE_K}, sim_thresh={SIM_THRESH}, algorithm={ALGORITHM}")
print(f"  appearance={APPEARANCE_WEIGHT}, hsv={HSV_WEIGHT}, spatiotemporal={ST_WEIGHT}")
print(f"  fic_reg={FIC_REG}, gallery={GALLERY_THRESH}, orphan={ORPHAN_MATCH_THRESH}")
print(f"  baseline DINOv2 weight at w_cs_ft=0.0: {BASELINE_DINOV2_WEIGHT}")
print(f"  sweep: {WEIGHT_SWEEP}")

fusion_results = []
for w_cs_ft in WEIGHT_SWEEP:
    label = f"clipsenetft_{int(round(w_cs_ft * 100)):03d}"
    run_name = f"{RUN_NAME}-{label}"
    ensure_upstream_links(run_name)
    cmd, w_primary, w_dino, w_clip_senet_ft = build_cmd(run_name, w_cs_ft)
    print("=" * 100)
    print(f"[{label}] w_primary={w_primary:.3f} w_dino={w_dino:.3f} w_cs_ft={w_clip_senet_ft:.3f}")
    print("CMD:", " ".join(str(part) for part in cmd))
    start = time.time()
    result = subprocess.run(cmd, cwd=str(PROJECT), capture_output=True, text=True)
    elapsed_min = (time.time() - start) / 60.0
    report_path = DATA_OUT / run_name / "stage5" / "evaluation_report.json"
    metrics = load_metrics(report_path)
    row = {
        "label": label,
        "run_name": run_name,
        "w_primary": w_primary,
        "w_dinov2": w_dino,
        "w_cs_ft": w_clip_senet_ft,
        "time_min": round(elapsed_min, 2),
        "returncode": result.returncode,
        "num_clusters": count_clusters(run_name),
        **metrics,
    }
    fusion_results.append(row)
    print(f"[{label}] returncode={result.returncode} time={elapsed_min:.1f} min metrics={metrics}")
    if result.returncode != 0:
        print("stdout tail:")
        print("\n".join(result.stdout.splitlines()[-40:]))
        print("stderr tail:")
        print("\n".join(result.stderr.splitlines()[-40:]))
        raise SystemExit(result.returncode)

print("Fusion sweep complete")

In [ ]:
results_path = WORK_DIR / "13h_fusion_results.json"
summary_path = WORK_DIR / "13h_summary.json"

results_payload = {
    "fusion_granularity": "tracklet",
    "baseline_note": "w_cs_ft=0.0 keeps the TransReID+DINOv2 score-fusion control with w_dinov2=0.60.",
    "clip_senet_tracklet_embeddings": str(CLIP_TRACKLET_EMBEDDINGS_PATH),
    "feature_manifest": "/kaggle/working/13h_13f_feature_manifest.json",
    "results": fusion_results,
}
results_path.write_text(json.dumps(results_payload, indent=2), encoding="utf-8")

summary = {
    "run_name": RUN_NAME,
    "kernel": "yahiaakhalafallah/13h-clip-senet-cityflow-fusion-ft",
    "fusion_granularity": "tracklet",
    "inputs": {
        "transreid_primary": str(PRIMARY_EMBEDDINGS_PATH),
        "dinov2_tertiary": str(TERTIARY_DINOV2_PATH),
        "clip_senet_13f_checkpoint": str(CHECKPOINT_PATH),
        "clip_senet_13f_per_detection": str(OUTPUT_DIR),
        "clip_senet_13f_tracklet": str(CLIP_TRACKLET_EMBEDDINGS_PATH),
    },
    "alignment_report": alignment_report,
    "weights": WEIGHT_SWEEP,
    "results_file": str(results_path),
    "feature_manifest": "/kaggle/working/13h_13f_feature_manifest.json",
}
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

results_df = pd.DataFrame(fusion_results)
display_columns = ["label", "w_primary", "w_dinov2", "w_cs_ft", "mtmc_idf1", "idf1", "mota", "hota", "conflations", "num_clusters", "time_min"]
existing_display_columns = [column for column in display_columns if column in results_df.columns]
print(f"Saved results: {results_path}")
print(f"Saved summary: {summary_path}")
display(results_df[existing_display_columns])

In [ ]:
best = max(fusion_results, key=lambda row: row.get("mtmc_idf1") if row.get("mtmc_idf1") is not None else -1.0)
control = next((row for row in fusion_results if abs(row["w_cs_ft"] - 0.0) < 1e-9), None)
control_value = control.get("mtmc_idf1") if control else None
best_value = best.get("mtmc_idf1")
delta = None if control_value is None or best_value is None else best_value - control_value

print("13h CLIP-SENet CityFlow fusion (FT) summary")
print(f"  checkpoint: {CHECKPOINT_PATH}")
print(f"  13f features: {CLIP_TRACKLET_EMBEDDINGS_PATH}")
print(f"  results: /kaggle/working/13h_fusion_results.json")
print(f"  summary: /kaggle/working/13h_summary.json")
print(f"  best: {best['label']} w_cs_ft={best['w_cs_ft']:.2f} MTMC_IDF1={best_value}")
if delta is not None:
    print(f"  delta_vs_control: {delta:+.6f}")
print(json.dumps(best, indent=2))